In [ ]:
!pip install -q "unstructured[pdf,docx,pptx]" unstructured-inference
# !pip install -q langchain langchain-community langchain-openai langchain-text-splitters
# !pip install -q faiss-cpu tiktoken python-dotenv
# !pip install -q pandas tabulate

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

from dotenv import load_dotenv
load_dotenv()

SAMPLE_DIR = "samples"
BATCH_DIR = os.path.join(SAMPLE_DIR, "batch")


In [2]:
from unstructured.partition.auto import partition
from collections import Counter

In [3]:
pdf_path = os.path.join(SAMPLE_DIR, 'business_report.pdf')

In [4]:
pdf_path

'samples/business_report.pdf'

In [5]:
elements = partition(filename = pdf_path)

No languages specified, defaulting to English.


In [6]:
len(elements)

73

In [10]:
for i, el in enumerate(elements[:5]):
    print(el.category,"|||",  el)

Title ||| 2024 Annual Business Report
Title ||| TechCorp International Inc.
Title ||| Executive Summary
NarrativeText ||| This report presents TechCorp International's financial performance for fiscal year 2024. The company achieved record revenue of $2.8 billion, representing a 23% year-over-year growth. Our AI and cloud services division drove the majority of this growth, contributing 65% of total revenue.
NarrativeText ||| Key strategic initiatives including the acquisition of DataFlow Systems and expansion into the Asia-Pacific market have positioned the company for sustained growth in 2025 and beyond.


In [11]:
from unstructured.partition.auto import partition
from unstructured.partition.pdf import partition_pdf
# partition_txt, partition_html

In [12]:
elements = partition_pdf(filename = pdf_path, strategy='fast')

No languages specified, defaulting to English.


In [14]:
e = elements[0]

In [15]:
type(e)

unstructured.documents.elements.Title

In [16]:
e.category, e.id

('Title', '70aab7bb68d429fd46386e78fdc8bba1')

In [18]:
e.metadata.page_number, e.metadata.filename, e.metadata.filetype

(1, 'business_report.pdf', 'application/pdf')

In [ ]:
# fast : text only
# hi-res : layout- detectron, yolox, table, image

In [19]:
elements_hi = partition_pdf(filename=pdf_path, strategy='hi_res', infer_table_structure=True)

No languages specified, defaulting to English.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

In [20]:
for c, n in Counter(el.category for el in elements).most_common():
    print(f" {c} : {n}")

 UncategorizedText : 33
 Title : 23
 NarrativeText : 9
 ListItem : 8


In [21]:
for c, n in Counter(el.category for el in elements_hi).most_common():
    print(f" {c} : {n}")

 Title : 10
 NarrativeText : 10
 Table : 2


In [ ]:
# Title -> semantic based chunking
# table -> 본문  별도로 인덱

In [ ]:
def analyze_document(pdf_path: str, strategy: str = "fast") -> dict:
    els = partition_pdf(filename=pdf_path, strategy=strategy)
    type_stats = dict(Counter(el.category for el in els))
    page_dist = defaultdict(lambda: Counter())
    for el in els:
        p = int(el.metadata.page_number or 0)
        page_dist[p][el.category] += 1
    narratives = [el for el in els if el.category == "NarrativeText"]
    narratives.sort(key=lambda e: len(str(e)), reverse=True)
    top3 = [{
        "length": len(str(el)),
        "page": int(el.metadata.page_number or 0),
        "text_preview": str(el)[:80],
    } for el in narratives[:3]]
    return {
        "total_elements": len(els),
        "type_statistics": type_stats,
        "page_distribution": {p: dict(c) for p, c in page_dist.items()},
        "top3_longest_narratives": top3,
    }

report = analyze_document(pdf_path)

In [25]:
# coordinates : title, text , table, image
meta_dict = elements[0].metadata.to_dict()
meta_dict

{'coordinates': {'points': ((128.2578, 67.66089999999997),
   (128.2578, 91.66089999999997),
   (467.0177999999999, 91.66089999999997),
   (467.0177999999999, 67.66089999999997)),
  'system': 'PixelSpace',
  'layout_width': 595.2756,
  'layout_height': 841.8898},
 'file_directory': 'samples',
 'filename': 'business_report.pdf',
 'last_modified': '2026-05-21T12:44:44',
 'page_number': 1,
 'languages': ['eng'],
 'filetype': 'application/pdf'}

In [26]:
from unstructured.partition.html import partition_html
from unstructured.partition.docx import partition_docx
from unstructured.partition.pptx import partition_pptx

In [32]:
html_path = os.path.join(SAMPLE_DIR, 'product_page.html')
docx_path = os.path.join(SAMPLE_DIR, 'quarterly_report.docx')
pptx_path = os.path.join(SAMPLE_DIR, 'investor_deck.pptx')

In [35]:
html_els = partition_html(filename = html_path)
docx_els = partition_docx(filename = docx_path)
pptx_els = partition_pptx(filename = pptx_path)

In [36]:
len(html_els), len(docx_els), len(pptx_els)

(14, 13, 11)

In [37]:
import pandas as pd

rows = []

for fmt, els in [('html', html_els), ('docx', docx_els), ('pptx', pptx_els)]:
    cats = Counter(el.category for el in els)
    rows.append({'format' : fmt, 'total' : len(els), **dict(cats)})

In [38]:
df = pd.DataFrame(rows).fillna(0).set_index('format')
print(df.to_string())

        total  Title  UncategorizedText  NarrativeText  ListItem  Table  PageBreak
format                                                                            
html       14      5                1.0              3       4.0      1        0.0
docx       13      5                1.0              2       4.0      1        0.0
pptx       11      7                0.0              1       0.0      1        2.0


In [ ]:
# from unstructured.partition.text import partition_text
# class MultiFormatParser:
#     PARSERS = {
#         '.pdf' : partition_pdf,
#         '.html' : partition_html,
#         '.txt' : partition_text
#     }

#     def __init__(self):
#         self.results = {}

#     def parse(self, path:str) -> list:
#         ext = os.path.splitext(path)[1].lower()
#         els = list(self.PARSERS[ext](filename=path))
#         self.results[path] = els
#         return els

In [42]:
from unstructured.chunking.title import chunk_by_title
from unstructured.chunking.basic import chunk_elements

In [43]:
chunks_title = chunk_by_title(
    elements,
    max_characters = 1000,
    new_after_n_chars = 800,
    combine_text_under_n_chars = 200
)

In [44]:
chunk_basic = chunk_elements(elements, max_characters=500, overlap=50)

In [48]:
str(chunks_title[0])[:200]

"2024 Annual Business Report\n\nTechCorp International Inc.\n\nExecutive Summary\n\nThis report presents TechCorp International's financial performance for fiscal year 2024. The company achieved record reven"

In [49]:
def adaptive_chunk(elements: list) -> list:
    if not elements:
        return []
    avg_len = sum(len(str(e)) for e in elements) / len(elements)
    if avg_len >=200:
        max_chars = 1000
    elif avg_len >= 50:
        max_chars = 500
    else:
        max_chars = 300

    return chunk_by_title(
        elements,
        max_characters = max_chars,
        combine_text_under_n_chars = 200
    )

In [50]:
from unstructured.cleaners.core import clean, clean_extra_whitespace, replace_unicode_quotes

In [51]:
samples = [
    "  This   is   a    test   with   extra   spaces.  ",
    "Revenue was “$2.8 billion” in FY2024.",
    "Line\n\n\n\nwith too many\nlinebreaks",
]

In [52]:
for s in samples:
    cleaned = clean_extra_whitespace(replace_unicode_quotes(s))
    print(cleaned)

This is a test with extra spaces.
Revenue was “$2.8 billion” in FY2024.
Line with too many linebreaks


In [53]:
from langchain_community.document_loaders import UnstructuredFileLoader

In [54]:
loader = UnstructuredFileLoader(pdf_path, mode = 'elements', strategy='fast')  # mode : single, paged

/tmp/ipykernel_560630/2883632221.py:1: LangChainDeprecationWarning: The class `UnstructuredFileLoader` was deprecated in LangChain 0.2.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-unstructured package and should be used instead. To use it run `pip install -U `langchain-unstructured` and import as `from `langchain_unstructured import UnstructuredLoader``.
  loader = UnstructuredFileLoader(pdf_path, mode = 'elements', strategy='fast')


In [55]:
documents = loader.load()

No languages specified, defaulting to English.


In [57]:
documents[:2]

[Document(metadata={'source': 'samples/business_report.pdf', 'coordinates': {'points': ((128.2578, 67.66089999999997), (128.2578, 91.66089999999997), (467.0177999999999, 91.66089999999997), (467.0177999999999, 67.66089999999997)), 'system': 'PixelSpace', 'layout_width': 595.2756, 'layout_height': 841.8898}, 'file_directory': 'samples', 'filename': 'business_report.pdf', 'last_modified': '2026-05-21T12:44:44', 'page_number': 1, 'languages': ['eng'], 'filetype': 'application/pdf', 'category': 'Title', 'element_id': '70aab7bb68d429fd46386e78fdc8bba1'}, page_content='2024 Annual Business Report'),
 Document(metadata={'source': 'samples/business_report.pdf', 'coordinates': {'points': ((213.2108, 117.59090000000003), (213.2108, 131.59090000000003), (382.06480000000005, 131.59090000000003), (382.06480000000005, 117.59090000000003)), 'system': 'PixelSpace', 'layout_width': 595.2756, 'layout_height': 841.8898}, 'file_directory': 'samples', 'filename': 'business_report.pdf', 'last_modified': '20

In [59]:
documents[0].metadata.items()

dict_items([('source', 'samples/business_report.pdf'), ('coordinates', {'points': ((128.2578, 67.66089999999997), (128.2578, 91.66089999999997), (467.0177999999999, 91.66089999999997), (467.0177999999999, 67.66089999999997)), 'system': 'PixelSpace', 'layout_width': 595.2756, 'layout_height': 841.8898}), ('file_directory', 'samples'), ('filename', 'business_report.pdf'), ('last_modified', '2026-05-21T12:44:44'), ('page_number', 1), ('languages', ['eng']), ('filetype', 'application/pdf'), ('category', 'Title'), ('element_id', '70aab7bb68d429fd46386e78fdc8bba1')])

In [60]:
for mode in ['single', 'elements', 'paged']:
    loader = UnstructuredFileLoader(pdf_path, mode = mode, strategy='fast')
    docs = loader.load()
    avg = sum(len(d.page_content) for d in docs) / max(len(docs),1)
    print(f"{mode} {len(docs)} {avg} {docs[0].page_content[:30]}")

No languages specified, defaulting to English.
No languages specified, defaulting to English.
No languages specified, defaulting to English.
`mode='paged'` is deprecated in favor of the 'by_page' chunking strategy. Learn more about chunking here: https://docs.unstructured.io/open-source/core-functionality/chunking


single 1 2814.0 2024 Annual Business Report

T
elements 73 36.57534246575342 2024 Annual Business Report
paged 3 938.6666666666666 2024 Annual Business Report

T


In [66]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
def answer_from_pdf(pdf_path, question, k=3) -> dict:

    # 1) pdf_path -> docs
    els = partition_pdf(filename=pdf_path, strategy='fast')
    cleaned = [el for el in els if el.category not in ['Header', 'Footer', 'PageBreak']]
    chunks = chunk_by_title(
        cleaned,
        max_characters = 600,
        combine_text_under_n_chars = 150,
    )
    docs = [
        Document(
            page_content = str(c),
            metadata = {
                'source' : os.path.basename(pdf_path),
                'page' : getattr(c.metadata, 'page_number', None),
                'category' : getattr(c.metadata, 'category', None)
            }
        ) for c in chunks if str(c).strip()
    ]
    
    # 2) embed + FAISS
    embeddings = OpenAIEmbeddings(model = 'text-embedding-3-small')
    vectorstore = FAISS.from_documents(docs, embeddings)

    # 3) retrieve
    hits = vectorstore.similarity_search(question, k=k)
    context = '\n\n'.join(f"[p{h.metadata.get('page', '?')}] {h.page_content}" for h in hits)
    
    # 4) generate
    llm = ChatOpenAI(model='gpt-4o-mini')
    msg = llm.invoke([SystemMessage(content = '주어진 컨텍스트에 근거해 답변하세요'), 
                HumanMessage(content = f'컨텍스트 :\n{context}\n\n질문: {question}')])

    return {
        'answer' : msg.content,
        'sources' : [{'page' : h.metadata.get('page', '?'), 'preview' : h.page_content[:30]} for h in hits]
    }

In [67]:
out = answer_from_pdf(pdf_path, 'what were the key financial highlights', k=3)

No languages specified, defaulting to English.


In [68]:
out

{'answer': "The key financial highlights for TechCorp International Inc. in fiscal year 2024 are as follows:\n\n- **Revenue**: $2.80 billion, a 23% increase from $2.28 billion in 2023.\n- **Net Income**: $420 million, up 35% from $310 million in the previous year.\n- **Operating Margin**: Improved to 18.5%, an increase of 3.3 percentage points from 15.2% in 2023.\n\nAdditionally, the company's AI and cloud services division significantly contributed to this growth, accounting for 65% of total revenue, and strategic initiatives such as the acquisition of DataFlow Systems and expansion into the Asia-Pacific market are expected to support further growth in 2025 and beyond.",
 'sources': [{'page': 1, 'preview': 'Financial Highlights\n\nMetric\n\n'},
  {'page': 1, 'preview': '2024 Annual Business Report\n\nT'},
  {'page': 2, 'preview': 'Revenue of $700M (+12% YoY). O'}]}